# User Tracking Dashboard

SystemLink Notebook datasource for Grafana.

This notebook:
- Reads UsageCalculationData and TargetPermissionHoldingsData from the Dataframe Service
- Applies Grafana time-picker range filtering
- Computes Casual Users, Standard Users, and Target Permission Users
- Applies target-permission priority so target-permission users are not counted as casual or standard

In [ ]:
# Parameters for Papermill or Grafana notebook execution
time_from = None
time_to = None
grafana_from = None
grafana_to = None
__from = None
__to = None

# Data source mode: auto | dataframe | files
Data_Source = "auto"
Server_Data_Dir = "C:/ProgramData/National Instruments/Shared"
Server_Usage_File = "usage_data.csv"
Server_Policy_File = "target_permission_holdings_data.csv"

# Bucket behavior
Include_Partial_Trailing_Month = False

# Standard user rule: minimum value_changes in rolling lookback window
Standard_User_Min_Logins = 25
Standard_User_Period_Months = 12

# Optional backward compatibility override (legacy variable name)
Standard_User_Threshold = None

# Target permission lookback window
Target_Permission_Period_Months = 12

# Projection horizon
Forecast_Months = 6

Casual_Users = 0
Standard_Users = 0
Target_Permission_Users = 0


In [ ]:
from datetime import datetime, timezone
from typing import Any, Iterable, List, Optional, Set

import os

import pandas as pd
import scrapbook as sb

try:
    from nisystemlink.clients.dataframe import DataFrameClient
    from nisystemlink.clients.dataframe.models import QueryTablesRequest, QueryTableDataRequest
except Exception:
    DataFrameClient = None
    QueryTablesRequest = None
    QueryTableDataRequest = None


def _parse_datetime_utc(value: Any, fallback: datetime) -> datetime:
    if value in (None, ""):
        return fallback

    try:
        if isinstance(value, (int, float)):
            # Grafana commonly passes unix epoch in milliseconds
            return datetime.fromtimestamp(float(value) / 1000.0, tz=timezone.utc)

        if isinstance(value, str):
            stripped = value.strip()
            if stripped.isdigit():
                return datetime.fromtimestamp(float(stripped) / 1000.0, tz=timezone.utc)
            ts = pd.to_datetime(stripped, utc=True)
            return ts.to_pydatetime()
    except Exception:
        pass

    return fallback


now_utc = datetime.now(timezone.utc)
default_from = now_utc - pd.Timedelta(days=30)

time_from_raw = time_from if time_from not in (None, "") else grafana_from
time_to_raw = time_to if time_to not in (None, "") else grafana_to

if time_from_raw in (None, ""):
    time_from_raw = __from
if time_to_raw in (None, ""):
    time_to_raw = __to

Data_Source = str(Data_Source).strip().lower()
Server_Data_Dir = str(Server_Data_Dir).strip()
Server_Usage_File = str(Server_Usage_File).strip()
Server_Policy_File = str(Server_Policy_File).strip()
Include_Partial_Trailing_Month = str(Include_Partial_Trailing_Month).strip().lower() in (
    "1",
    "true",
    "yes",
    "y",
)

if Data_Source not in ("auto", "dataframe", "files"):
    raise ValueError("Data_Source must be one of: auto, dataframe, files")
if not Server_Data_Dir:
    raise ValueError("Server_Data_Dir must not be empty")
if not Server_Usage_File:
    raise ValueError("Server_Usage_File must not be empty")
if not Server_Policy_File:
    raise ValueError("Server_Policy_File must not be empty")

# Backward compatibility for legacy threshold parameter.
if Standard_User_Threshold not in (None, ""):
    Standard_User_Min_Logins = Standard_User_Threshold

Standard_User_Min_Logins = int(Standard_User_Min_Logins)
Standard_User_Period_Months = int(Standard_User_Period_Months)
Target_Permission_Period_Months = int(Target_Permission_Period_Months)
Forecast_Months = int(Forecast_Months)

Casual_Users = int(Casual_Users)
Standard_Users = int(Standard_Users)
Target_Permission_Users = int(Target_Permission_Users)

if Standard_User_Min_Logins < 1:
    raise ValueError("Standard_User_Min_Logins must be >= 1")
if Standard_User_Period_Months < 1:
    raise ValueError("Standard_User_Period_Months must be >= 1")
if Target_Permission_Period_Months < 1:
    raise ValueError("Target_Permission_Period_Months must be >= 1")
if Forecast_Months < 1:
    raise ValueError("Forecast_Months must be >= 1")

time_from = _parse_datetime_utc(time_from_raw, default_from)
time_to = _parse_datetime_utc(time_to_raw, now_utc)

if time_to < time_from:
    time_from, time_to = time_to, time_from

print(f"Grafana time range: {time_from.isoformat()} to {time_to.isoformat()}")
print(f"Data Source Mode: {Data_Source}")
print(f"Server Data Dir: {Server_Data_Dir}")
print(f"Include Partial Trailing Month: {Include_Partial_Trailing_Month}")
print(f"Standard User Min Logins: {Standard_User_Min_Logins}")
print(f"Standard User Period Months: {Standard_User_Period_Months}")
print(f"Target Permission Period Months: {Target_Permission_Period_Months}")
print(f"Forecast Months: {Forecast_Months}")


In [ ]:
def _extract_column_names(columns_obj: Iterable[Any]) -> List[str]:
    names: List[str] = []
    for col in columns_obj:
        if isinstance(col, str):
            names.append(col)
        elif hasattr(col, "name"):
            names.append(getattr(col, "name"))
    return names


def _frame_to_df(frame: Any) -> pd.DataFrame:
    columns = _extract_column_names(getattr(frame, "columns", []))
    data = getattr(frame, "data", [])

    if not isinstance(data, list):
        data = []

    if columns and data:
        return pd.DataFrame(data=data, columns=columns)

    if columns and not data:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(data=data)


def _query_table_rows(
    client: Any,
    table_id: str,
    query_from: datetime,
    query_to: datetime,
) -> pd.DataFrame:
    if QueryTableDataRequest is None or client is None or not hasattr(client, "query_table_data"):
        raise RuntimeError("DataFrameClient.query_table_data is not available in this runtime.")

    last_error: Optional[Exception] = None
    filter_expr = f'day >= "{query_from.isoformat()}" AND day <= "{query_to.isoformat()}"'

    request_attempts = [
        {"filter": filter_expr, "take": 50000},
        {"where": filter_expr, "take": 50000},
        {"filter": filter_expr},
        {},
    ]

    for kwargs in request_attempts:
        try:
            req = QueryTableDataRequest(**kwargs)
            resp = client.query_table_data(table_id, req)

            frame = getattr(resp, "frame", None)
            if frame is None and hasattr(resp, "data"):
                frame = getattr(resp, "data")
            if frame is None and hasattr(resp, "data_frame"):
                frame = getattr(resp, "data_frame")

            if frame is not None:
                return _frame_to_df(frame)
        except Exception as ex:
            last_error = ex

    if last_error is not None:
        raise last_error

    raise RuntimeError("DataFrameClient.query_table_data did not return frame data.")


def _get_table_id_by_name(client: Any, table_name: str) -> str:
    if QueryTablesRequest is None or client is None or not hasattr(client, "query_tables"):
        raise RuntimeError("DataFrameClient.query_tables is not available in this runtime.")

    table_query = QueryTablesRequest(filter=f'name = "{table_name}"')
    table_details = client.query_tables(table_query)

    if len(table_details.tables) == 0:
        raise RuntimeError(f"No dataframe named '{table_name}' was found.")
    if len(table_details.tables) > 1:
        raise RuntimeError(f"Multiple dataframes named '{table_name}' were found.")

    return table_details.tables[0].id


def load_dataframe_by_name(
    client: Any,
    table_name: str,
    query_from: datetime,
    query_to: datetime,
) -> pd.DataFrame:
    table_id = _get_table_id_by_name(client, table_name)
    df = _query_table_rows(client, table_id, query_from, query_to)

    if "day" in df.columns:
        df["day"] = pd.to_datetime(df["day"], utc=True, errors="coerce")
        df = df[(df["day"] >= query_from) & (df["day"] <= query_to)]

    return df


def load_dataframe_from_file(csv_path: str, query_from: datetime, query_to: datetime) -> pd.DataFrame:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path, sep=",", dtype=str)
    if "day" in df.columns:
        df["day"] = pd.to_datetime(df["day"], utc=True, errors="coerce")
        df = df[(df["day"] >= query_from) & (df["day"] <= query_to)]

    return df.fillna("")


def load_table_with_fallback(
    client: Any,
    table_name: str,
    query_from: datetime,
    query_to: datetime,
    data_source: str,
    server_data_dir: str,
) -> pd.DataFrame:
    csv_name_map = {
        "UsageCalculationData": Server_Usage_File,
        "TargetPermissionHoldingsData": Server_Policy_File,
    }
    if table_name not in csv_name_map:
        raise ValueError(f"Unsupported table name: {table_name}")

    csv_path = os.path.join(server_data_dir, csv_name_map[table_name])

    if data_source == "files":
        return load_dataframe_from_file(csv_path, query_from, query_to)

    if data_source == "dataframe":
        return load_dataframe_by_name(client, table_name, query_from, query_to)

    dataframe_error: Optional[Exception] = None
    file_error: Optional[Exception] = None

    try:
        return load_dataframe_by_name(client, table_name, query_from, query_to)
    except Exception as ex:
        dataframe_error = ex
        print(f"DataFrame read failed for {table_name}: {ex}")
        print(f"Attempting file fallback: {csv_path}")

    try:
        return load_dataframe_from_file(csv_path, query_from, query_to)
    except Exception as ex:
        file_error = ex

    raise RuntimeError(
        f"Failed to load {table_name} from both sources. "
        f"DataFrame error: {dataframe_error}. "
        f"File error: {file_error}."
    )


selected_month_start = time_from.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
max_lookback_months = max(Standard_User_Period_Months, Target_Permission_Period_Months)
preload_from = (pd.Timestamp(selected_month_start) - pd.DateOffset(months=max_lookback_months)).to_pydatetime()

client = DataFrameClient() if DataFrameClient is not None else None
usage_df = load_table_with_fallback(
    client,
    "UsageCalculationData",
    preload_from,
    time_to,
    Data_Source,
    Server_Data_Dir,
)
policy_df = load_table_with_fallback(
    client,
    "TargetPermissionHoldingsData",
    preload_from,
    time_to,
    Data_Source,
    Server_Data_Dir,
)

print(f"Preload range: {preload_from.isoformat()} to {time_to.isoformat()}")
print(f"Selected range: {time_from.isoformat()} to {time_to.isoformat()}")
print(f"UsageCalculationData rows loaded: {len(usage_df)}")
print(f"TargetPermissionHoldingsData rows loaded: {len(policy_df)}")

In [ ]:
def _user_columns(df: pd.DataFrame) -> List[str]:
    return [column for column in df.columns if isinstance(column, str) and column.startswith("User_")]


def _month_start(dt: datetime) -> datetime:
    return dt.replace(day=1, hour=0, minute=0, second=0, microsecond=0)


def _next_month(dt: datetime) -> datetime:
    if dt.month == 12:
        return dt.replace(year=dt.year + 1, month=1)
    return dt.replace(month=dt.month + 1)


def _classify_month(
    usage_df: pd.DataFrame,
    policy_df: pd.DataFrame,
    month_start: datetime,
    month_end: datetime,
    standard_min_logins: int,
    standard_lookback_months: int,
    target_lookback_months: int,
) -> dict:
    usage_user_cols = _user_columns(usage_df)
    policy_user_cols = _user_columns(policy_df)

    if "day" in usage_df.columns:
        usage_month = usage_df[(usage_df["day"] >= month_start) & (usage_df["day"] < month_end)]
        standard_lookback_start = month_end - pd.DateOffset(months=standard_lookback_months)
        usage_standard_window = usage_df[
            (usage_df["day"] >= standard_lookback_start) & (usage_df["day"] < month_end)
        ]
    else:
        usage_month = usage_df
        usage_standard_window = usage_df

    if "day" in policy_df.columns:
        target_lookback_start = month_end - pd.DateOffset(months=target_lookback_months)
        policy_target_window = policy_df[
            (policy_df["day"] >= target_lookback_start) & (policy_df["day"] < month_end)
        ]
    else:
        policy_target_window = policy_df

    # Users with any activity in the current month.
    active_users: Set[str] = set()
    for column in usage_user_cols:
        if column not in usage_month.columns:
            continue
        has_activity = usage_month[column].fillna("").astype(str).str.strip().ne("").any()
        if has_activity:
            active_users.add(column)

    # Standard: active users with enough value_changes in rolling lookback window.
    standard_candidates: Set[str] = set()
    for column in active_users:
        if column not in usage_standard_window.columns:
            continue
        series = usage_standard_window[column].fillna("").astype(str)
        value_changes = max(0, int((series != series.shift(1)).sum() - 1))
        if value_changes >= standard_min_logins:
            standard_candidates.add(column)

    # Target-permission: any non-empty hit in rolling lookback window.
    target_permission_users: Set[str] = set()
    for column in policy_user_cols:
        if column not in policy_target_window.columns:
            continue
        non_empty_hits = int(
            policy_target_window[column].fillna("").astype(str).str.strip().ne("").sum()
        )
        if non_empty_hits > 0:
            target_permission_users.add(column)

    # Priority: target-permission wins.
    standard_final = standard_candidates - target_permission_users
    casual_final = (active_users - standard_candidates) - target_permission_users

    return {
        "time": month_start,
        "Casual Users": len(casual_final),
        "Standard Users": len(standard_final),
        "Target Permission Users": len(target_permission_users),
    }


def _linear_forecast_int(values: List[int], steps: int) -> List[int]:
    # Least-squares trend fit on month index; clamp negatives to zero.
    y = pd.Series(values, dtype="float64").fillna(0.0)
    n = len(y)
    if n == 0:
        return [0] * steps
    if n == 1:
        return [max(0, int(round(y.iloc[0])))] * steps

    x = pd.Series(range(n), dtype="float64")
    x_mean = x.mean()
    y_mean = y.mean()

    denom = ((x - x_mean) ** 2).sum()
    slope = 0.0 if denom == 0 else float(((x - x_mean) * (y - y_mean)).sum() / denom)
    intercept = float(y_mean - slope * x_mean)

    predictions: List[int] = []
    for k in range(steps):
        x_future = n + k
        y_hat = intercept + slope * x_future
        predictions.append(max(0, int(round(y_hat))))
    return predictions


usage_df = usage_df.copy()
policy_df = policy_df.copy()

if "day" in usage_df.columns:
    usage_df["day"] = pd.to_datetime(usage_df["day"], utc=True, errors="coerce")
if "day" in policy_df.columns:
    policy_df["day"] = pd.to_datetime(policy_df["day"], utc=True, errors="coerce")

# Build list of month boundaries within the selected time range.
months = []
cursor = _month_start(time_from)
month_loop_end = time_to if Include_Partial_Trailing_Month else _month_start(time_to)
while cursor < month_loop_end:
    months.append(cursor)
    cursor = _next_month(cursor)

monthly_rows = []
for month_start in months:
    month_end = _next_month(month_start)
    row = _classify_month(
        usage_df,
        policy_df,
        month_start,
        month_end,
        Standard_User_Min_Logins,
        Standard_User_Period_Months,
        Target_Permission_Period_Months,
    )
    monthly_rows.append(row)

historical_df = pd.DataFrame(
    monthly_rows,
    columns=["time", "Casual Users", "Standard Users", "Target Permission Users"],
)
historical_df["time"] = pd.to_datetime(historical_df["time"], utc=True)
historical_df["is_projection"] = False

# Non-seasonal projection: fit trend from recent history and project configurable future months.
TREND_WINDOW_MONTHS = 6
metric_columns = ["Casual Users", "Standard Users", "Target Permission Users"]

projection_rows = []
if len(historical_df) > 0:
    last_time = historical_df["time"].max()
    future_times = [last_time + pd.DateOffset(months=i) for i in range(1, Forecast_Months + 1)]

    projected_values = {}
    for metric in metric_columns:
        recent_history = historical_df[metric].tail(TREND_WINDOW_MONTHS).astype(int).tolist()
        projected_values[metric] = _linear_forecast_int(recent_history, Forecast_Months)

    for i, future_time in enumerate(future_times):
        projection_rows.append(
            {
                "time": pd.to_datetime(future_time, utc=True),
                "Casual Users": projected_values["Casual Users"][i],
                "Standard Users": projected_values["Standard Users"][i],
                "Target Permission Users": projected_values["Target Permission Users"][i],
                "is_projection": True,
            }
        )

projection_df = pd.DataFrame(
    projection_rows,
    columns=["time", "Casual Users", "Standard Users", "Target Permission Users", "is_projection"],
)

output_df = pd.concat([historical_df, projection_df], ignore_index=True)

print(output_df.to_string(index=False))
output_df


In [ ]:
def build_SystemLink_data_frame(df: pd.DataFrame):
    output_dict={
        'columns': pd.io.json.build_table_schema(df, index=False)['fields'],
        'values':df.values.tolist()
        }  
    return output_dict

In [ ]:
output_to_glue = output_df.copy()
if "is_projection" in output_to_glue.columns:
    # Keep this field simple for downstream table schema handling.
    output_to_glue["is_projection"] = output_to_glue["is_projection"].astype(int)

df_dict = build_SystemLink_data_frame(output_to_glue)

df_output = {
    'type': 'data_frame',
    'id': 'usage_tracking',
    'data': df_dict
}

result = [df_output]
sb.glue('result', result)


In [ ]:
# Validation: monthly breakdown
print("Monthly user counts:")
print(output_df.to_string(index=False))
output_df
